Data Extraction

In [3]:
import pandas as pd
from google_play_scraper import Sort, reviews

# Corrected Google Play Package IDs
apps = {
    'Blinkit': 'com.grofers.customerapp',
    'Zepto': 'com.zeptoconsumerapp'  # Fixed ID
}

all_reviews = []

for app_name, app_id in apps.items():
    print(f"Scraping reviews for {app_name}...")
    result, _ = reviews(
        app_id,
        lang='en',        # English reviews
        country='in',     # India store
        sort=Sort.NEWEST, # Latest reviews
        count=1500        # 1,500 reviews per app
    )
    
    for r in result:
        all_reviews.append({
            'app_name': app_name,
            'user_name': r['userName'],
            'rating': r['score'],
            'review_text': r['content'],
            'review_time': r['at'],
            'app_version': r['reviewCreatedVersion']
        })

# Save combined dataset
df = pd.DataFrame(all_reviews)
df.to_csv('quick_commerce_reviews_raw.csv', index=False)

print("\n--- Scraping Complete ---")
print("Total rows collected:", len(df))
print(df['app_name'].value_counts())

Scraping reviews for Blinkit...
Scraping reviews for Zepto...

--- Scraping Complete ---
Total rows collected: 3000
app_name
Blinkit    1500
Zepto      1500
Name: count, dtype: int64


Data Cleaning & Preprocessing

In [5]:
import pandas as pd
# 1. Load the raw dataset
df = pd.read_csv('quick_commerce_reviews_raw.csv')

# Verify exact app counts
print("Reviews scraped per app:")
print(df['app_name'].value_counts())

# 2. Clean text: remove null reviews & convert to lowercase
df = df.dropna(subset=['review_text'])
df['cleaned_text'] = df['review_text'].astype(str).str.lower()

# 3. Apply rule-based tagging for quick commerce churn drivers
def categorize_review(text):
    if any(k in text for k in ['delay', 'late', 'time', 'slow', 'waiting', 'delivery boy', 'minutes', 'rider']):
        return 'SLA & Delivery Delay'
    elif any(k in text for k in ['missing', 'wrong', 'damaged', 'spoiled', 'item', 'quality', 'rotten', 'expired']):
        return 'Order Accuracy & Quality'
    elif any(k in text for k in ['crash', 'bug', 'froze', 'otp', 'payment', 'location', 'map', 'update', 'login']):
        return 'App UI & Tech Bug'
    elif any(k in text for k in ['charge', 'price', 'expensive', 'surge', 'refund', 'money', 'cost', 'cashback']):
        return 'Pricing & Refund Issue'
    else:
        return 'General / Other'

df['issue_category'] = df['cleaned_text'].apply(categorize_review)

# 4. Save cleaned dataset for SQL loading
df.to_csv('quick_commerce_reviews_cleaned.csv', index=False)
print("\n--- Categorization Complete ---")
print(df.groupby(['app_name', 'issue_category']).size())

Reviews scraped per app:
app_name
Blinkit    1500
Zepto      1500
Name: count, dtype: int64

--- Categorization Complete ---
app_name  issue_category          
Blinkit   App UI & Tech Bug             19
          General / Other             1234
          Order Accuracy & Quality      67
          Pricing & Refund Issue        73
          SLA & Delivery Delay         107
Zepto     App UI & Tech Bug             23
          General / Other              996
          Order Accuracy & Quality     189
          Pricing & Refund Issue        85
          SLA & Delivery Delay         207
dtype: int64
